In [ ]:
!pip uninstall -y langchain langchain-community
!pip install -q --upgrade langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu  pandas

Found existing installation: langchain 1.3.11
Uninstalling langchain-1.3.11:
  Successfully uninstalled langchain-1.3.11
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.4 MB/s eta 0:00:00
ERROR: pip's 

In [ ]:
!pip install -U bitsandbytes accelerate transformers

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/tmp/ipykernel_3561/647856499.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
import os
import torch
import bitsandbytes as bnb # Explicitly import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# Set environment variable to help with CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name, fix_mistral_regex=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/181k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/29.9k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# --- STEP 1: LOAD STRUCTURED CSV (No splitting needed, rows are natural chunks) ---
csv_loader = CSVLoader(file_path="defacto_inventory_100.csv")
csv_docs = csv_loader.load()

# --- STEP 2: LOAD UNSTRUCTURED TXT (Splitting is REQUIRED) ---
txt_loader = TextLoader("policy.txt")
txt_docs = txt_loader.load()

# Split only the text documents into logical paragraphs
text_splitter = RecursiveCharacterTextSplitter(chunk_size=550, chunk_overlap=100)
txt_chunks = text_splitter.split_documents(txt_docs)

# --- STEP 3: MERGE BOTH LISTS ---
# csv_docs contains individual rows, txt_chunks contains individual paragraphs
final_combined_chunks = csv_docs + txt_chunks

# --- STEP 4: EMBED AND INDEX EVERYTHING TOGETHER ---
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)

# This builds one unified database containing both products and store rules
vectordb = FAISS.from_documents(final_combined_chunks, embedding)

print(f"✅ Multi-source Ingestion Complete!")
print(f"Total structured product rows indexed: {len(csv_docs)}")
print(f"Total unstructured policy chunks indexed: {len(txt_chunks)}")

/tmp/ipykernel_3561/580665668.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Multi-source Ingestion Complete!
Total structured product rows indexed: 100
Total unstructured policy chunks indexed: 25


In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

# 1. Create a native Hugging Face text-generation pipeline using your current model & tokenizer
hf_text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,     # ⚡ Increased for full generation capacity
    temperature=0.1,        # 🔒 Forces quick, direct factual answers
    do_sample=False,        # 🚫 Turned off sampling to prevent conversational looping
    pad_token_id=tokenizer.eos_token_id
)

# 2. Wrap it for LangChain
llm = HuggingFacePipeline(pipeline=hf_text_pipeline)
print("✅ Local Hugging Face model successfully wrapped into LangChain!")

✅ Local Hugging Face model successfully wrapped into LangChain!


In [ ]:
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Turn your existing FAISS database into a retriever that fetches the top 3 matches
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

# --- MEMORY HANDLER ---
# Rephrases follow-up questions contextually so FAISS search doesn't fail
context_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the chat history and the user's latest query. Turn it into a standalone question if it references past items, otherwise return it unchanged."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
history_aware_retriever = create_history_aware_retriever(llm, retriever, context_prompt)

# --- SYSTEM BOUNDS QA PROMPT ---
# Restricts your open-source model to strictly talk about DeFacto Egypt data
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are the virtual customer care assistant for DeFacto Egypt.\n"
        "Answer the customer's question accurately using ONLY the retrieved context pieces below. "
        "If an item is out of stock, say so. If you do not know the answer, say 'I am sorry, I cannot find that in our store database.'\n\n"
        "Context:\n{context}"
    )),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Combine everything together into a conversational RAG pipeline
document_synthesis_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_bot = create_retrieval_chain(history_aware_retriever, document_synthesis_chain)

print("✅ Conversational RAG Pipeline is fully assembled and live!")

✅ Conversational RAG Pipeline is fully assembled and live!


In [ ]:
from collections import deque

# This will automatically maintain a maximum of 3 items
conversation_history = deque(maxlen=3)
memory_store = {}

In [ ]:
from collections import deque
from langchain_core.messages import HumanMessage, AIMessage

# 2 messages per turn * 3 turns = maxlen of 6
chat_history = deque(maxlen=6)

evaluation_queries = [
    "What items do you have under the Accessories category?",
    "How much does that leather look belt cost, and what aisle is it located in?",
    "If I buy it online using cash on delivery, what extra fees apply in Egypt?",
   # "Can I return it back to a physical store if it doesn't fit me?"
]

print("🚀 --- Running Midterm Project Evaluation Execution --- \n")

for turn, query in enumerate(evaluation_queries, start=1):
    print(f"Turn {turn} | Customer: {query}")

    # Pass the history converted to a standard list
    response = rag_bot.invoke({
        "input": query,
        "chat_history": list(chat_history)
    })

    print(f"Turn {turn} | DeFacto Bot: {response['answer']}\n" + "="*60 + "\n")

    # The deque automatically drops the 2 oldest messages when these 2 are added past the limit
    chat_history.extend([
        HumanMessage(content=query),
        AIMessage(content=response['answer'])
    ])

    # 2. Append the new turn to the history array
    chat_history.extend([
        HumanMessage(content=query),
        AIMessage(content=response['answer'])
    ])

🚀 --- Running Midterm Project Evaluation Execution --- 

Turn 1 | Customer: What items do you have under the Accessories category?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Turn 1 | DeFacto Bot: System: You are the virtual customer care assistant for DeFacto Egypt.
Answer the customer's question accurately using ONLY the retrieved context pieces below. If an item is out of stock, say so. If you do not know the answer, say 'I am sorry, I cannot find that in our store database.'

Context:
Item_ID: DF-30331
Item_Name: Canvas Backpack with Laptop Sleeve (Black)
Category: Accessories
Price_EGP: 699
Size_Range: 28-36
Stock_Status: Low Stock
Stock_Quantity: 15
Aisle_Location: Aisle D-7

Item_ID: DF-64522
Item_Name: Stainless Steel Minimalist Watch (Grey Melange)
Category: Accessories
Price_EGP: 299
Size_Range: 36-44
Stock_Status: In Stock
Stock_Quantity: 45
Aisle_Location: Aisle D-11
Human: What items do you have under the Accessories category?

Turn 2 | Customer: How much does that leather look belt cost, and what aisle is it located in?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Turn 2 | DeFacto Bot: System: You are the virtual customer care assistant for DeFacto Egypt.
Answer the customer's question accurately using ONLY the retrieved context pieces below. If an item is out of stock, say so. If you do not know the answer, say 'I am sorry, I cannot find that in our store database.'

Context:
Item_ID: DF-30331
Item_Name: Canvas Backpack with Laptop Sleeve (Black)
Category: Accessories
Price_EGP: 699
Size_Range: 28-36
Stock_Status: Low Stock
Stock_Quantity: 15
Aisle_Location: Aisle D-7

Item_ID: DF-30639
Item_Name: Cargo Trousers with Pocket Details (Grey Melange)
Category: Men's Wear
Price_EGP: 1999
Size_Range: S-XXL
Stock_Status: In Stock
Stock_Quantity: 50
Aisle_Location: Aisle B-4
Human: What items do you have under the Accessories category?
AI: System: You are the virtual customer care assistant for DeFacto Egypt.
Answer the customer's question accurately using ONLY the retrieved context pieces below. If an item is out of stock, say so. If you do not know t

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Turn 3 | DeFacto Bot: System: You are the virtual customer care assistant for DeFacto Egypt.
Answer the customer's question accurately using ONLY the retrieved context pieces below. If an item is out of stock, say so. If you do not know the answer, say 'I am sorry, I cannot find that in our store database.'

Context:
Item_ID: DF-19733
Item_Name: Leather Bi-Fold Wallet (Black)
Category: Accessories
Price_EGP: 999
Size_Range: Standard
Stock_Status: In Stock
Stock_Quantity: 127
Aisle_Location: Aisle D-1

Item_ID: DF-30331
Item_Name: Canvas Backpack with Laptop Sleeve (Black)
Category: Accessories
Price_EGP: 699
Size_Range: 28-36
Stock_Status: Low Stock
Stock_Quantity: 15
Aisle_Location: Aisle D-7
Human: What items do you have under the Accessories category?
AI: System: You are the virtual customer care assistant for DeFacto Egypt.
Answer the customer's question accurately using ONLY the retrieved context pieces below. If an item is out of stock, say so. If you do not know the answer, say 

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from collections import deque

def print_chat_history(history_deque: deque, num_messages_to_show: int = None):
    print("\n" + "=" * 60)
    if num_messages_to_show is None:
        print(f"FULL CHAT HISTORY ({len(history_deque)} MESSAGES)")
    else:
        print(f"LAST {min(num_messages_to_show, len(history_deque))} MESSAGES")
    print("=" * 60)

    if not history_deque:
        print("No history.")
        return

    messages_to_display = history_deque
    if num_messages_to_show is not None:
        # Show the last 'num_messages_to_show' messages
        messages_to_display = list(history_deque)[-num_messages_to_show:]

    for msg in messages_to_display:
        role = "Customer" if isinstance(msg, HumanMessage) else "Bot"
        # Truncate content to avoid overwhelming output
        print(f"  [{role}] {str(msg.content)[:80]}{'...' if len(str(msg.content)) > 80 else ''}")

    print("=" * 60)

In [ ]:
import gc
import torch

# 1. Clear Python's hidden references
gc.collect()

# 2. Release idle cached VRAM back to the GPU
torch.cuda.empty_cache()

print(f"Allocated VRAM: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"Reserved/Cached VRAM: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

In [ ]:
# Rerun the evaluation loop after applying fixes and clearing history
# Expected: No OutOfMemoryError, and chat_history should be managed correctly.
import re
from collections import deque
from langchain_core.messages import HumanMessage, AIMessage

import warnings
import os # Import os for os.devnull
import sys # Import sys for stderr redirection

# Suppress specific warnings from transformers and bitsandbytes
warnings.filterwarnings("ignore", category=FutureWarning) # For bitsandbytes
warnings.filterwarnings("ignore", category=UserWarning, message="Both `max_new_tokens` *=* and `max_length` *=* seem to have been set.") # For transformers warning


print("🚀 --- Running Midterm Project Evaluation Execution (with fixes) --- \n")

# Reinitialize chat_history to ensure it's clean for this run, in case the previous clearing was temporary.
chat_history = deque(maxlen=6)

for turn, query in enumerate(evaluation_queries, start=1):
    print(f"Turn {turn} | Customer: {query}")

    response = rag_bot.invoke({
        "input": query,
        "chat_history": list(chat_history)
    })

    bot_raw_response = response['answer']

    # Extract only the actual answer from the bot's raw response
    # The model output format seems to be System:...Context:...Human:...AI: <ACTUAL_ANSWER>
    last_ai_index = bot_raw_response.rfind('AI:')
    if last_ai_index != -1:
        # Extract everything after the last 'AI:'
        actual_bot_answer = bot_raw_response[last_ai_index + len('AI:'):].strip()
        if not actual_bot_answer:
            actual_bot_answer = "(Bot started to respond but generated no content - check max_new_tokens)"
    else:
        # Fallback if 'AI:' is not found at all, implying generation was cut off
        actual_bot_answer = "(Bot did not generate 'AI:' token. Max_new_tokens might be too low or unexpected output format.)"

    print(f"Turn {turn} | DeFacto Bot: {actual_bot_answer}\n" + "="*60 + "\n")

    chat_history.extend([
        HumanMessage(content=query),
        AIMessage(content=actual_bot_answer)
    ])

    print(f"Memory: {len(chat_history)} messages ({len(chat_history)//2} turns)")
    for msg in chat_history:
        role = "Customer" if isinstance(msg, HumanMessage) else "Bot"
        print(f"  [{role}] {msg.content[:80]}")
    print("="*60 + "\n")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🚀 --- Running Midterm Project Evaluation Execution (with fixes) --- 

Turn 1 | Customer: What items do you have under the Accessories category?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Turn 1 | DeFacto Bot: (Bot did not generate 'AI:' token. Max_new_tokens might be too low or unexpected output format.)

Memory: 2 messages (1 turns)
  [Customer] What items do you have under the Accessories category?
  [Bot] (Bot did not generate 'AI:' token. Max_new_tokens might be too low or unexpected

Turn 2 | Customer: How much does that leather look belt cost, and what aisle is it located in?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Turn 2 | DeFacto Bot: ' token. Max_new_tokens might be too low or unexpected output format.)
Human: How much does that leather look belt cost, and what aisle is it located in?

Memory: 4 messages (2 turns)
  [Customer] What items do you have under the Accessories category?
  [Bot] (Bot did not generate 'AI:' token. Max_new_tokens might be too low or unexpected
  [Customer] How much does that leather look belt cost, and what aisle is it located in?
  [Bot] ' token. Max_new_tokens might be too low or unexpected output format.)
Human: Ho

Turn 3 | Customer: If I buy it online using cash on delivery, what extra fees apply in Egypt?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Turn 3 | DeFacto Bot: ' token. Max_new_tokens might be too low or unexpected output format.)
Human: How much does that leather look belt cost, and what aisle is it located in?
Human: If I buy it online using cash on delivery, what extra fees apply in Egypt?

Memory: 6 messages (3 turns)
  [Customer] What items do you have under the Accessories category?
  [Bot] (Bot did not generate 'AI:' token. Max_new_tokens might be too low or unexpected
  [Customer] How much does that leather look belt cost, and what aisle is it located in?
  [Bot] ' token. Max_new_tokens might be too low or unexpected output format.)
Human: Ho
  [Customer] If I buy it online using cash on delivery, what extra fees apply in Egypt?
  [Bot] ' token. Max_new_tokens might be too low or unexpected output format.)
Human: Ho



In [ ]:
# 1. Clear the chat history
chat_history.clear()

# 2. Verify it's empty
print_chat_history(chat_history)


FULL CHAT HISTORY (0 MESSAGES)
No history.
